In [ ]:
import sys
from pathlib import Path
import os

parent_folder = Path(os.getcwd()).parent  # parent of current folder
sys.path.insert(0, str(parent_folder))

import torch

from PhaseDataset import PhaseDataset
from SimuEnd2EndWFS import End2EndWFS, CheckpointManager
from LossFunctions import LogResidualVarianceLoss
from MaskGeneration import MaskVisualizator
from mmengine import Config
import numpy as np
import matplotlib.pyplot as plt
import time

from IPython.display import display, clear_output


from tqdm import tqdm

In [ ]:
def train_closed_loop(End2EndWFS, dataset, TrainRunNb, optimizer_n, num_iterations=5, device='cuda'):
    
    gain = (torch.rand((dataset.Nphases, 1), device = device) * 0.5 + 0.2 )
    leak = (torch.rand((dataset.Nphases, 1), device = device) * 0.05 + 0.95 )
    
    if visual: maskVisualizator.SetCanvas()
    
    z_FullRes = dataset.z_FullRes.to(device, dtype=torch.float32).view(-1, dataset.Nmodes).transpose(0, 1)

    End2EndWFS.train()
    
    progressBar = tqdm(range(TrainRunNb // num_iterations))
    
    for u in progressBar:

        dataset.ResetMovingWavefront()
        phaseGT,pupilGT,modes,photons,ron,r0s,_,_ = dataset[0]
        End2EndWFS.WFS.SetPhotonsAndRON(photons, ron)

        # Closed-loop correction
        z_estimated = torch.zeros_like(modes)  # Start with zero correction 
        z_buffer = torch.zeros_like(modes)  
        z_output = torch.zeros_like(modes)    
        z_reconstructed = torch.zeros_like(phaseGT)
        z_reconstructed_iter = torch.zeros_like(phaseGT)
        z_reconstructed_ideal = torch.zeros_like(phaseGT)
        z_reconstructed_iter_ideal = torch.zeros_like(phaseGT)
        z_prev = torch.zeros_like(modes)

        total_loss = 0
        ideal_loss = 0

        for i in range(num_iterations):
            # Get new WFS images after applying the correction
            if i > 0:
                phaseGT,pupilGT, modes,_,_,_,_,_ = dataset[i]
            residual_phase = phaseGT - z_reconstructed
            Ze = torch.matmul(residual_phase.flatten(1,2), dataset.invZ)

            # Predict coefficients and update estimate
            z_estimated = z_estimated * leak + gain * z_buffer  # Apply correction with gain
            z_buffer = torch.clone(z_output)
            z_output = End2EndWFS(residual_phase, pupilGT)
            
            # Convert modes coefficients to full-resolution wavefront
            z_reconstructed = torch.matmul(z_estimated, z_FullRes).view_as(phaseGT)
            z_reconstructed_iter = torch.matmul(z_output, z_FullRes).view_as(phaseGT)
            
            z_reconstructed_ideal = torch.matmul(modes, z_FullRes).view_as(phaseGT)
            z_reconstructed_iter_ideal = torch.matmul(Ze, z_FullRes).view_as(phaseGT)
            
            
            # Compute loss for this iteration
            total_loss += loss_variance((residual_phase - z_reconstructed_iter)) / num_iterations

            # Compute ideal loss for comparison
            with torch.no_grad():
                ideal_loss += loss_variance((residual_phase - z_reconstructed_iter_ideal)) / num_iterations

            
        # **Backpropagation**
        optimizer_n.zero_grad(set_to_none = True)
        
        total_loss.backward()
            
        optimizer_n.step()
        
        # **Track loss and parameters**
        
        loss_tracker[u] = total_loss.detach()
        loss_tracker_ideal[u] = ideal_loss.detach()
        
        if u % (300 // num_iterations) == 1:
            progressBar.set_postfix({'Loss': float(total_loss), 'Loss_ideal': float(ideal_loss)})
            if visual: 
                maskVisualizator.update_plots(Ze, z_output)
                maskVisualizator.show()
           




In [ ]:
@torch.no_grad() 
def closeLoop():
    
    gain = 0.3
    dataset.Nphases = 1
    dataset.ResetMovingWavefront()
    phaseGT,pupilGT,modes,photons,ron,r0s,_,_ = dataset[0]
    Trained_End2EndWFS.WFS.SetPhotonsAndRON(photons, ron)

    Trained_End2EndWFS.eval()
    z_output = Trained_End2EndWFS(phaseGT, pupilGT)
    

    fig, ax = plt.subplots(1,5, figsize=(24,18))
    img0 = ax[0].imshow(phaseGT[0].cpu().detach())
    img1 = ax[1].imshow(phaseGT[0].cpu().detach())
    img2 = ax[2].imshow(phaseGT[0].cpu().detach())
    img3 = ax[3].imshow(phaseGT[0].cpu().detach())
    img4 = ax[4].imshow(Trained_End2EndWFS.Image[0].cpu().detach())
    

    ax[0].set_title('Pupil function')
    ax[1].set_title('Input phase')
    ax[2].set_title('Reconstructed phase')
    ax[3].set_title('Residual')
    ax[4].set_title('WFS frame')
    
    a = time.perf_counter()

    z_output = Trained_End2EndWFS(phaseGT, pupilGT)
            
    # Closed-loop correction
    z_estimated = torch.zeros_like(z_output)  # Start with zero correction 
    z_reconstructed = torch.zeros_like(phaseGT)    
    z_buffer = torch.zeros_like(modes)  
    z_output = torch.zeros_like(modes)      
    
    for i in range(100):
        a = time.perf_counter()
        # Get new WFS images after applying the correction
        if i > 0:
            phaseGT,pupilGT,modes,_,_,_,_,_ = dataset[i]
        # residual_phase = phaseGT - z_reconstructed
        residual_phase = phaseGT - z_reconstructed 
        

        z_estimated = z_estimated * 0.999 + gain * z_buffer  # Apply correction with gain
        z_buffer = torch.clone(z_output)
        z_output = Trained_End2EndWFS(residual_phase, pupilGT)
        

        z_reconstructed = torch.matmul(z_estimated, z_FullRes).view_as(phaseGT)    

        
        clear_output(wait=True)
        img0.set_data((pupilGT[0] * dataset.pupil).cpu().detach().numpy())
        img0.set_clim(vmin=np.min(img0.get_array()), vmax=np.max(img0.get_array()))
        img1.set_data(phaseGT[0].cpu().detach().numpy())
        img1.set_clim(vmin=np.min(img1.get_array()), vmax=np.max(img1.get_array()))
        img2.set_data(z_reconstructed[0].cpu().detach().numpy())
        img2.set_clim(vmin=np.min(img1.get_array()), vmax=np.max(img1.get_array()))
        img3.set_data(residual_phase[0].cpu().detach().numpy())
        img3.set_clim(vmin=np.min(img1.get_array()), vmax=np.max(img1.get_array()))
        img4.set_data(Trained_End2EndWFS.Image[0].cpu().detach().numpy())
        img4.set_clim(vmin=np.min(img4.get_array()), vmax=np.max(img4.get_array()))
        display(fig, clear=True)
        
        
        #fig.canvas.draw()
        #fig.canvas.flush_events()
        plt.pause(0.001)
        
        b = time.perf_counter()
        
        print(f"Loop frequency = {1 / (b - a):.1f} Hz. Residual variance = {Trained_End2EndWFS.GetPhaseVariance(residual_phase[0]).item():.2f} ", end="\r", flush=True)


In [ ]:
device = 'cuda' # set to "cpu" if Cuda is not available
    
paramfile = 'train_params_exp.py'  # file of experimental parameters


# Config extraction
AtmosParams = Config.fromfile(paramfile)['AtmosParams']
WFSParams = Config.fromfile(paramfile)['WFSParams']
LoopParams = Config.fromfile(paramfile)['LoopParams']
TrainParams = Config.fromfile(paramfile)['TrainParams']

# Dataset creation
dataset = PhaseDataset(WFSParams, AtmosParams, LoopParams, device)
z_FullRes = dataset.z_FullRes.to(device, dtype=torch.float32).view(-1, dataset.Nmodes).transpose(0, 1)


# Initialisation of the system 
Trained_End2EndWFS = End2EndWFS(WFSParams, AtmosParams, device)

# Setting the loss function
loss_variance = LogResidualVarianceLoss(dataset.pupil)

# Optimization parameters (learning rate lr and nb of runs)
lrn = TrainParams['lrn']
lro = TrainParams['lro']

# Number of training and testing run 
TrainRunNb = TrainParams['TrainRunNb']


# Setting the optimizer (here Adam)
optimizer_o = torch.optim.AdamW(Trained_End2EndWFS.maskManager.parameters(), lro, fused = True)
#optimizer_n = torch.optim.AdamW(Trained_End2EndWFS.PhaseEstimator.parameters(), lrn, fused = True, weight_decay=1e-2)
optimizer_n = torch.optim.AdamW(Trained_End2EndWFS.PhaseEstimator.parameters(), lrn, fused = True)
# optimizer_n = torch.optim.AdamW(Trained_End2EndWFS.PhaseEstimator.papy2.parameters(), lrn, fused = True)

checkpoint_path = "../Data/Train_test.pth"
# checkpoint_path = "../Data/RAMA5.pth"
checkpointManager = CheckpointManager(Trained_End2EndWFS, WFSParams, TrainParams, checkpoint_path, optimizer_o, optimizer_n)
checkpointManager.load()


Trained_End2EndWFS.UpdateMask()
Trained_End2EndWFS.OptimizeMask = TrainParams["OptimizeMask"]

# Training part for parameters optimization
num_iterations = 1
loss_tracker = torch.zeros(TrainRunNb // num_iterations, device=device)
loss_tracker_ideal = torch.zeros(TrainRunNb // num_iterations, device=device)

maskVisualizator = MaskVisualizator(Trained_End2EndWFS, loss_tracker, loss_tracker_ideal)
visual = True


total_params = sum(p.numel() for p in Trained_End2EndWFS.PhaseEstimator.parameters() if p.requires_grad)
print(f"Total trainable parameters: {total_params:,}")




In [ ]:
train_closed_loop(Trained_End2EndWFS,
                    dataset,
                    TrainRunNb,
                    optimizer_n,
                    num_iterations=num_iterations,
                    device=device)

In [ ]:
checkpointManager.save()

In [ ]:
closeLoop()